# QRShield v2 — 2/4: Frequency and DINOv2 Models, Final + Leave-One-Family-Out

For each of 3 seeds, `freq` (gradient boosting) and `dino` (MLP on frozen features) are trained
* **final**: on all families, tested in-distribution and on the camera-degraded benchmark;
* **LOFO**: one tamper family held out of training *and* validation, tested on that family (+ benign test images).

**Protocol notes**
* The decision threshold is chosen on the validation set (never on test). In LOFO the validation set also excludes the held-out family.
* The only DINOv2 design choice (token pooling) is made on **validation** LOFO AUROC and then frozen (cell below).
* For these two methods, the seed only changes the classifier (features are deterministic), so their std reflects classifier variance, not feature-extractor variance.

Run notebook 1 first.

In [ ]:
# ── Common setup (identical in all four notebooks) ───────────────────────────
import os, gc, sys, json, time, glob, shutil, subprocess, warnings, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # not running on Colab

SMOKE = True          # True: quick pipeline check on a small slice (separate output folders). Set False for the full study.
SMOKE_ROWS = 3000     # number of manifest rows used in smoke mode
SEEDS = [0, 1, 2]
IMG = 224
FAMILIES = ['sticker_overlay', 'module_corruption', 'patch_overlay', 'logo_manipulation', 'occlusion_warp']
V1_FAMILIES = ['module_corruption', 'patch_overlay']        # the two tamper families v1 was trained on
METHODS = ['v1', 'aug', 'v2', 'freq', 'dino']

ROOT = Path(os.environ.get('QR_ROOT', '/content/drive/MyDrive/qrshield_v2'))
D = Path(os.environ.get('QR_DATA', '/content/qrshield_v2_data'))          # local copy of the dataset
ZIPS = Path(os.environ.get('QR_ZIPS', '/content/zips'))
OUT = ROOT / ('v2_eval_smoke' if SMOKE else 'v2_eval')
FEA, RES, MOD, TAB = (OUT / n for n in ('feats', 'results', 'models', 'tables'))
for p in (FEA, RES, MOD, TAB):
    p.mkdir(parents=True, exist_ok=True)
print(f'SMOKE={SMOKE} | outputs -> {OUT}')

In [ ]:
# ── Data: copy shards to the local disk, extract, merge the manifest ─────────
def prepare_data():
    if (D / '.done').exists():
        return
    D.mkdir(parents=True, exist_ok=True)
    ZIPS.mkdir(parents=True, exist_ok=True)
    src = sorted(glob.glob(str(ROOT / 'shard500' / 'shard_*.zip')))
    assert src, f'No shard_*.zip found in {ROOT / "shard500"}'
    for z in src:                                   # bulk copy Drive -> local disk (fast)
        dst = ZIPS / Path(z).name
        if not dst.exists():
            shutil.copy(z, dst)
    zips = sorted(glob.glob(str(ZIPS / 'shard_*.zip')))
    def unz(z):
        subprocess.run(['unzip', '-q', '-o', z, '-x', 'manifest.csv', '-d', str(D)], check=True)
    with ThreadPoolExecutor(2) as ex:
        list(ex.map(unz, zips))
    parts = []
    for z in zips:
        with zipfile.ZipFile(z) as zz:
            parts.append(pd.read_csv(zz.open('manifest.csv'), keep_default_na=False))
    merged = pd.concat(parts, ignore_index=True)
    merged.to_csv(D / 'manifest.csv', index=False)
    shutil.rmtree(ZIPS, ignore_errors=True)         # free local disk
    (D / '.done').write_text('ok')
    print('shards:', len(zips), '| manifest rows:', len(merged))

prepare_data()

man = pd.read_csv(D / 'manifest.csv', keep_default_na=False)
if SMOKE:
    man = man.iloc[:SMOKE_ROWS].copy()
need = ['image', 'set', 'split', 'base_id', 'cls', 'family', 'camera']
miss = [c for c in need if c not in man.columns]
assert not miss, f'manifest is missing columns {miss}; found {man.columns.tolist()}'

man['fp'] = str(D) + '/' + man['image'].astype(str)
man['cls_'] = man['cls'].astype(str)
man['label'] = (~man.cls_.str.startswith('benign')).astype(int)
man['fam'] = np.where(man.label == 1, man.cls_.str.replace('^tamper_', '', regex=True), 'benign')
man['base'] = man['base_id'].astype(str)
man['split_'] = man['split'].astype(str)
man['set_'] = man['set'].astype(str)
man['has_cam'] = man['camera'].astype(str).str.strip().str.lower().eq('true')
man['fidx'] = np.arange(len(man))

missing = [p for p in man.fp if not os.path.exists(p)]
assert not missing, f'{len(missing)} image files missing, e.g. {missing[:3]}'
assert set(man.fam.unique()) - {'benign'} == set(FAMILIES), man.fam.unique()

main = man[man.set_ == 'main']
tr = main[main.split_ == 'train']
va = main[main.split_ == 'val']
te = main[main.split_ == 'test']
cam = man[man.set_ == 'camera_bench']
# integrity checks: group split (no base QR in two splits); benchmark only from test bases
assert not set(tr.base) & set(te.base) and not set(tr.base) & set(va.base) and not set(va.base) & set(te.base)
assert not set(cam.base) & set(tr.base) and not set(cam.base) & set(va.base)
share = main.groupby('label').has_cam.mean()
assert 0.3 < share.min() and share.max() < 0.5, f'camera-layer share should be ~0.4 for both labels: {share.to_dict()}'
print(f'rows={len(man)} | main={len(main)} (train {len(tr)} / val {len(va)} / test {len(te)}) | camera_bench={len(cam)}')
print('camera-layer share by label:', share.round(3).to_dict())

In [ ]:
# ── Metrics and result storage ────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, balanced_accuracy_score

def pick_th(y, p):
    """Decision threshold chosen on the validation set (maximises balanced accuracy)."""
    ths = np.linspace(0.02, 0.98, 97)
    return float(ths[int(np.argmax([balanced_accuracy_score(y, p > t) for t in ths]))])

def m_row(y, p, th):
    y = np.asarray(y)
    return dict(n=int(len(y)), acc=accuracy_score(y, p > th), bal_acc=balanced_accuracy_score(y, p > th),
                auroc=roc_auc_score(y, p) if len(set(y)) > 1 else np.nan,
                f1=f1_score(y, p > th, zero_division=0), acc05=accuracy_score(y, p > 0.5))

def family_records(df, p, th, test, meta):
    """One record for ALL images plus one per tamper family. For a family, the benign images are
    subsampled (fixed seed) to the same size, so accuracy / F1 are not distorted by class imbalance."""
    df = df.reset_index(drop=True)
    ben = np.where(df.label.values == 0)[0]
    groups = [('ALL', np.arange(len(df)))]
    for f in sorted(df.fam[df.label == 1].unique()):
        idx = np.where(df.fam.values == f)[0]
        b = np.random.default_rng(123).choice(ben, size=min(len(idx), len(ben)), replace=False)
        groups.append((f, np.concatenate([b, idx])))
    return [dict(meta, test=test, family=f, th=th, **m_row(df.label.values[i], p[i], th)) for f, i in groups]

def rpath(key): return RES / f'{key}.json'
def done(key): return rpath(key).exists()

def finish(key, score, th, tests, meta):
    recs = []
    for name, df in tests.items():
        recs += family_records(df, score(df), th, name, meta)
    tmp = RES / f'{key}.tmp'
    json.dump(recs, open(tmp, 'w'))
    os.replace(tmp, rpath(key))                      # atomic: a half-written result never counts as done

def tests_final():
    return {'indist': te, 'camera_bench': cam}

def tests_lofo(held):
    return {'indist_heldout': pd.concat([te[te.label == 0], te[te.fam == held]]),
            'camera_heldout': pd.concat([cam[cam.label == 0], cam[cam.fam == held]])}

def v1_subset(df):
    """v1-style training data: plain benign + the two v1 tamper families, no camera layer."""
    return df[(df.fam.isin(V1_FAMILIES) | (df.cls_ == 'benign_plain')) & ~df.has_cam]

def show_status():
    names = [f.stem for f in RES.glob('*__s*.json')]
    print('finished jobs per method:', pd.Series([n.split('__')[0] for n in names]).value_counts().to_dict())

## Features and classifiers

In [ ]:
# ── Load features ─────────────────────────────────────────────────────────────
DINO_TAG = 'dino_small'
FREQ_X = np.concatenate([np.load(f) for f in sorted((FEA / 'freq').glob('part_*.npy')) if '.tmp' not in f.name])
_dino = np.concatenate([np.load(f) for f in sorted((FEA / DINO_TAG).glob('part_*.npy')) if '.tmp' not in f.name])
assert len(FREQ_X) == len(_dino) == len(man), 'Run notebook 1 first (feature rows must match the manifest).'
BLK = _dino.shape[1] // 4                     # blocks: 0=CLS, 1=mean, 2=max, 3=std
FEATURE_SETS = {'cls+mean': [0, 1], 'cls+mean+max': [0, 1, 2], 'all4': [0, 1, 2, 3]}

def dino_X(name):
    cols = np.concatenate([np.arange(b * BLK, (b + 1) * BLK) for b in FEATURE_SETS[name]])
    return _dino[:, cols].astype(np.float32)
print('freq', FREQ_X.shape, '| dino blocks of', BLK)

In [ ]:
# ── Classifiers ───────────────────────────────────────────────────────────────
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def make_clf(method, seed):
    if method == 'freq':
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, early_stopping=True, random_state=seed)
    return make_pipeline(StandardScaler(),
                         MLPClassifier((256,), early_stopping=True, max_iter=100, random_state=seed))

## DINOv2 design choice (validation only)

In [ ]:
# ── DINOv2 design choice, made on VALIDATION data only ────────────────────────
# Which token pooling to use is chosen by the mean validation AUROC over leave-one-family-out splits
# (train without family f, validate on the *validation* images of f + benign). The test set is never touched.
sel_file = OUT / 'dino_config.json'
if sel_file.exists():
    cfg = json.load(open(sel_file))
else:
    scores = {}
    for name in FEATURE_SETS:
        X = dino_X(name)
        aucs = []
        for f in FAMILIES:
            trn = tr[tr.fam != f]
            vf = va[(va.fam == f) | (va.label == 0)]
            clf = make_clf('dino', 0).fit(X[trn.fidx.values], trn.label.values)
            aucs.append(roc_auc_score(vf.label.values, clf.predict_proba(X[vf.fidx.values])[:, 1]))
        scores[name] = float(np.mean(aucs))
        print(f'{name:14s} mean val-LOFO AUROC = {scores[name]:.4f}', flush=True)
    cfg = dict(feature_set=max(scores, key=scores.get), val_lofo_auroc=scores)
    json.dump(cfg, open(sel_file, 'w'), indent=2)
print('DINOv2 configuration frozen:', cfg)
DINO_X = dino_X(cfg['feature_set'])

## Final models and LOFO, 3 seeds
Re-run this cell if it stops; finished jobs are skipped.

In [ ]:
# ── Jobs ──────────────────────────────────────────────────────────────────────
def feat_job(method, setting, seed):
    key = f'{method}__{setting}__s{seed}'
    if done(key):
        return
    held = setting[5:] if setting.startswith('lofo_') else None
    X = FREQ_X if method == 'freq' else DINO_X
    tr_, va_ = (tr, va) if held is None else (tr[tr.fam != held], va[va.fam != held])
    clf = make_clf(method, seed).fit(X[tr_.fidx.values], tr_.label.values)
    score = lambda df: clf.predict_proba(X[df.fidx.values])[:, 1]
    th = pick_th(va_.label.values, score(va_))
    tests = tests_final() if held is None else tests_lofo(held)
    meta = dict(method=method, setting=setting, seed=seed)
    if method == 'dino':
        meta['variant'] = cfg['feature_set']
    finish(key, score, th, tests, meta)

def job_list():
    return [(m, s, seed) for seed in SEEDS for m in ('freq', 'dino')
            for s in ['final'] + [f'lofo_{f}' for f in FAMILIES]]

def run_queue(max_minutes=120):
    t0 = time.time()
    todo = [j for j in job_list() if not done(f'{j[0]}__{j[1]}__s{j[2]}')]
    print(f'{len(todo)} of {len(job_list())} jobs remaining')
    for j in todo:
        if (time.time() - t0) / 60 > max_minutes:
            print('time budget reached; run this cell again to continue'); return
        t1 = time.time()
        feat_job(*j)
        print(f'  done {j} ({(time.time() - t1) / 60:.1f} min)', flush=True)
    print('queue complete')

run_queue()
show_status()